# 03 · Recuperación: línea base léxica, sistema propuesto y ablación

**Laboratorio 4 · Asistente de compras por voz** (MCD8016 Deep Learning, UTEC)

El enunciado pide *"comparar la recuperación con un modelo base de búsqueda léxica y realizar una ablación sobre un componente del sistema, utilizando las mismas consultas"*, y reportar **nDCG@10 sobre los candidatos anotados**, con ganancias E = 1, S = 0,1, C = 0,01 e I = 0.

**Métodos comparados** (código en `src/asistente_compras/retrieval.py`):

| Método | Rol | Idea |
|---|---|---|
| BM25 | **Línea base léxica** | Coincidencia de palabras con stemming en español. Fuerte con marcas y modelos |
| Denso (e5) | Componente | `multilingual-e5-base`: compara significados, no palabras ("zapatillas" ≈ "deportivas") |
| Híbrido (RRF) | Primera etapa | Combina las posiciones de BM25 y de e5 |
| **Híbrido + re-ranker** | **Sistema propuesto** | `bge-reranker-v2-m3` (cross-encoder) re-ordena los 50 primeros leyendo consulta y producto juntos |
| Sin re-ranker | **Ablación** | El sistema sin su último componente (equivale al híbrido) |

**Disciplina de evaluación:**
1. Todas las decisiones (campos indexados, configuración) se toman con las **200 consultas de desarrollo** (split *train*).
2. La configuración se **congela** de forma programática, eligiendo el máximo en desarrollo.
3. Las **250 consultas de test** se evalúan **una sola vez** al final.

## 0. Entorno

In [ ]:
# ── Configuración del entorno (idéntica en todos los notebooks) ──────────────────────────
# Detectar si corre en Colab o en la Mac y dejar importable el paquete `asistente_compras`.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # 1) Montar Drive: guarda el código y los datos entre sesiones (Colab borra /content al cerrar).
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/lab4/lab4_solution")
    os.environ["LAB4_DATA_DIR"] = "/content/drive/MyDrive/lab4/data"
    # 2) Instalar el paquete y sus dependencias. EXTRAS cambia según el notebook
    #    ("" = núcleo; "rag", "llm", "voz" en los siguientes).
    EXTRAS = "rag"
    target = f"{PROJECT_DIR}[{EXTRAS}]" if EXTRAS else str(PROJECT_DIR)
    get_ipython().run_line_magic("pip", f"install -q -e {target}")
else:
    # En la Mac, usar como kernel el entorno uv del proyecto (ver README).
    PROJECT_DIR = Path.cwd().resolve().parent

# Una instalación editable recién hecha no se ve hasta reiniciar el kernel;
# agregar src/ al path para evitar ese reinicio.
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"Colab: {EN_COLAB} | proyecto: {PROJECT_DIR}")

In [2]:
import numpy as np
import pandas as pd

from asistente_compras.config import get_paths
from asistente_compras.evaluation import (
    compare, evaluate_candidates, load_qrels, per_query_table, recall_exact_at_k, timed,
)
from asistente_compras.metrics import ndcg_at_k
from asistente_compras.retrieval import (
    FIELD_SETS, BM25Retriever, DenseRetriever, HybridRetriever, RerankRetriever, TextConfig,
    pick_device, product_texts,
)

pd.set_option("display.precision", 4)
paths = get_paths().ensure()
catalog = pd.read_parquet(paths.processed / "catalog.parquet")
q_dev = pd.read_parquet(paths.processed / "queries_dev.parquet")
q_test = pd.read_parquet(paths.processed / "queries_test.parquet")
# Cargar las etiquetas solo aquí, para medir; ningún método las ve al ordenar.
qrels_dev, qrels_test = load_qrels(paths, "dev"), load_qrels(paths, "test")
print(f"Catálogo {len(catalog):,} | dev {len(q_dev)} | test {len(q_test)} | dispositivo: {pick_device()}")

Catálogo 11,463 | dev 200 | test 250 | dispositivo: cuda


## 1. Referencia: ¿cuánto da un orden al azar?

Como se re-ordenan **solo candidatos anotados** y el 41 % de ellos son Exact, un orden aleatorio ya obtiene un nDCG@10 alto. Sin este piso, los números de los métodos se malinterpretan: la escala útil va de ≈ 0,54 (azar) a 1,0 (orden ideal).

In [3]:
def ndcg_aleatorio(qrels, reps=20, seed=0):
    rng = np.random.default_rng(seed)
    return np.mean([np.mean([ndcg_at_k(list(rng.permutation(list(qr))), qr) for _ in range(reps)])
                    for qr in qrels.values()])

print(f"nDCG@10 aleatorio · dev: {ndcg_aleatorio(qrels_dev):.4f} | test: {ndcg_aleatorio(qrels_test):.4f}")

nDCG@10 aleatorio · dev: 0.5436 | test: 0.5451


## 2. Ajuste en desarrollo (200 consultas *train*)

### 2.1 ¿Qué campos del producto indexar con BM25?
Se prueban cuatro combinaciones (`FIELD_SETS`). Los campos vacíos se omiten y cada documento se trunca a 200 palabras.

In [4]:
dev_bm25 = {}
for fields in FIELD_SETS:
    cfg = TextConfig(fields=fields)
    r = BM25Retriever(catalog.product_id, product_texts(catalog, cfg))
    dev_bm25[cfg] = evaluate_candidates(r, q_dev, qrels_dev)["ndcg@10"].mean()
pd.Series({c.fields: v for c, v in dev_bm25.items()}, name="nDCG@10 dev (BM25)").to_frame()

,nDCG@10 dev (BM25)
titulo,0.6775
titulo_marca,0.6853
titulo_marca_bullets,0.6757
completo,0.6699


### 2.2 ¿Qué campos usar en el modelo denso?
Codificar el catálogo cuesta ~1–3 min por configuración (en GPU o MPS), así que se prueban las dos más prometedoras. Los embeddings quedan en caché en `data/index/*.npy` y no se recalculan en las siguientes ejecuciones.

In [5]:
dev_dense, dense_models = {}, {}
for fields in ("titulo_marca", "titulo_marca_bullets"):
    cfg = TextConfig(fields=fields)
    d, secs = timed(DenseRetriever, catalog.product_id, product_texts(catalog, cfg),
                    cache_dir=paths.index, cache_key=cfg.key())
    dense_models[cfg] = d
    dev_dense[cfg] = evaluate_candidates(d, q_dev, qrels_dev)["ndcg@10"].mean()
    print(f"{fields:22s} nDCG@10 dev = {dev_dense[cfg]:.4f}  (carga/codificación {secs:.0f} s)")

titulo_marca           nDCG@10 dev = 0.7322  (carga/codificación 44 s)


titulo_marca_bullets   nDCG@10 dev = 0.7412  (carga/codificación 12 s)


### 2.3 Congelar la configuración
Se elige el máximo en desarrollo de cada método. **A partir de aquí no se cambia nada.**

In [6]:
CFG_BM25 = max(dev_bm25, key=dev_bm25.get)
CFG_DENSE = max(dev_dense, key=dev_dense.get)
print("BM25  →", CFG_BM25)
print("Denso →", CFG_DENSE)

bm25 = BM25Retriever(catalog.product_id, product_texts(catalog, CFG_BM25))
dense = dense_models[CFG_DENSE]
hybrid = HybridRetriever(bm25, dense)
# Dar al re-ranker el texto más rico (con bullets), hasta 384 tokens por par consulta–producto.
reranker, carga_rr = timed(RerankRetriever, hybrid, catalog.product_id,
                           product_texts(catalog, CFG_DENSE))
print(f"Carga del re-ranker: {carga_rr:.1f} s")

BM25  → TextConfig(fields='titulo_marca', max_words=200)
Denso → TextConfig(fields='titulo_marca_bullets', max_words=200)


Carga del re-ranker: 20.4 s


### 2.4 Comprobación en desarrollo del sistema completo
Antes de tocar test se verifica en desarrollo que cada componente aporte. Es la única ejecución larga del notebook: el re-ranker tarda ~1–2 s por consulta en la Mac y bastante menos en una GPU de Colab.

In [7]:
METODOS = {"BM25": bm25, "Denso (e5)": dense, "Híbrido (RRF)": hybrid, "Híbrido + re-ranker": reranker}
res_dev = {name: evaluate_candidates(m, q_dev, qrels_dev) for name, m in METODOS.items()}
compare(res_dev, baseline="BM25")

,ndcg@10,IC95 inf,IC95 sup,Δ vs BM25,gana / empata / pierde,p (Wilcoxon),ms/consulta
método,,,,,,,
BM25,0.6853,0.6538,0.7183,0.0000,0 / 200 / 0,NaN,0.2014
Denso (e5),0.7412,0.7101,0.7695,0.0559,120 / 14 / 66,4.2537e-05,14.9760
Híbrido (RRF),0.7276,0.6973,0.7591,0.0424,121 / 18 / 61,4.2939e-07,15.5129
Híbrido + re-ranker,0.7740,0.7425,0.8030,0.0887,137 / 17 / 46,8.9986e-12,137.7350


## 3. Evaluación única en test (250 consultas)

Misma configuración congelada, mismas consultas para todos los métodos. La tabla incluye:
- **IC 95 %**: intervalo bootstrap del promedio.
- **gana / empata / pierde**: número de consultas en que el método supera, iguala o queda por debajo de BM25.
- **p (Wilcoxon)**: prueba pareada consulta a consulta frente a BM25.

In [8]:
res_test = {name: evaluate_candidates(m, q_test, qrels_test) for name, m in METODOS.items()}
tabla_test = compare(res_test, baseline="BM25")
tabla_test

,ndcg@10,IC95 inf,IC95 sup,Δ vs BM25,gana / empata / pierde,p (Wilcoxon),ms/consulta
método,,,,,,,
BM25,0.7043,0.6769,0.7325,0.0000,0 / 250 / 0,NaN,0.2029
Denso (e5),0.7433,0.7165,0.7666,0.0390,144 / 16 / 90,2.5037e-04,15.2575
Híbrido (RRF),0.7357,0.7099,0.7612,0.0314,141 / 32 / 77,1.0451e-05,16.8713
Híbrido + re-ranker,0.7884,0.7647,0.8127,0.0842,157 / 29 / 64,3.7031e-12,143.8231


### 3.1 Ablación: el sistema con y sin re-ranker
El componente retirado es el re-ranker. Sin él, el sistema queda como la primera etapa (híbrido).

In [9]:
abl = compare({"Sistema completo": res_test["Híbrido + re-ranker"],
               "Sin re-ranker (ablación)": res_test["Híbrido (RRF)"]},
              baseline="Sin re-ranker (ablación)")
abl

,ndcg@10,IC95 inf,IC95 sup,Δ vs Sin re-ranker (ablación),gana / empata / pierde,p (Wilcoxon),ms/consulta
método,,,,,,,
Sistema completo,0.7884,0.7632,0.8122,0.0528,143 / 27 / 80,3.2277e-07,143.8231
Sin re-ranker (ablación),0.7357,0.7083,0.7605,0.0000,0 / 250 / 0,NaN,16.8713


### 3.2 Métrica complementaria: búsqueda en todo el catálogo
El nDCG oficial re-ordena candidatos ya conocidos, pero el agente **busca en los 11 463 productos**. Aquí se mide qué fracción de los productos *Exact* de cada consulta aparece entre los 50 primeros (*Recall@50*). Esta es la métrica que justifica usar el **híbrido como primera etapa**.

In [10]:
rec = {name: recall_exact_at_k(m, q_test, qrels_test, k=50)
       for name, m in {"BM25": bm25, "Denso (e5)": dense, "Híbrido (RRF)": hybrid}.items()}
pd.Series(rec, name="Recall@50 de Exact (catálogo completo, test)").to_frame()

,"Recall@50 de Exact (catálogo completo, test)"
BM25,0.7816
Denso (e5),0.7648
Híbrido (RRF),0.8231


### 3.3 Latencia: carga inicial frente a consultas
El enunciado pide distinguir la carga de los modelos de las consultas posteriores. Aquí se mide solo la etapa de recuperación; el notebook de voz suma ASR, LLM y TTS.

In [11]:
consulta = q_test["query"].iloc[0]
_, t_hyb = timed(hybrid.search, consulta, 50)
_, t_rr = timed(reranker.search, consulta, 10)
pd.DataFrame({
    "etapa": ["carga re-ranker (una vez)", "búsqueda híbrida top-50", "búsqueda + re-ranker top-10"],
    "segundos": [carga_rr, t_hyb, t_rr],
}).set_index("etapa")

,segundos
etapa,
carga re-ranker (una vez),20.3973
búsqueda híbrida top-50,0.0190
búsqueda + re-ranker top-10,0.3727


## 4. Análisis por consulta

Se guardan los resultados por consulta para el análisis de aciertos y fallos del informe final. Abajo se muestran las consultas donde el sistema más gana y más pierde frente a BM25.

In [12]:
pq = per_query_table(res_test)
pq["Δ sistema − BM25"] = pq["Híbrido + re-ranker"] - pq["BM25"]
pq.to_parquet(paths.index / "retrieval_test_por_consulta.parquet", index=False)
tabla_test.to_csv(paths.index / "retrieval_test_resumen.csv")
cols = ["query", "n_cand", "BM25", "Híbrido + re-ranker", "Δ sistema − BM25"]
print("Donde más GANA el sistema:")
display(pq.nlargest(5, "Δ sistema − BM25")[cols])
print("Donde más PIERDE el sistema:")
display(pq.nsmallest(5, "Δ sistema − BM25")[cols])

Donde más GANA el sistema:


,n_cand,BM25,Híbrido + re-ranker,Δ sistema − BM25
244,16,0.1386,1.0000,0.8614
134,40,0.0391,0.8459,0.8068
231,40,0.1515,0.9139,0.7624
182,39,0.2384,1.0000,0.7616
6,40,0.1000,0.8154,0.7154


Donde más PIERDE el sistema:


,n_cand,BM25,Híbrido + re-ranker,Δ sistema − BM25
41,37,0.6863,0.3599,-0.3264
138,15,1.0000,0.6934,-0.3066
193,39,1.0000,0.7023,-0.2977
228,32,1.0000,0.7046,-0.2954
210,40,0.7607,0.4908,-0.2700


In [ ]:
# Ver un ejemplo concreto: primeros 3 productos de cada método para la consulta con mayor ganancia.
qid = int(pq.nlargest(1, "Δ sistema − BM25").query_id.iloc[0])
titulos = catalog.set_index("product_id").title
etiqueta = {g: l for l, g in {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}.items()}
print("Consulta:", q_test.set_index("query_id").loc[qid, "query"])
for name in ("BM25", "Híbrido + re-ranker"):
    top = res_test[name].set_index("query_id").loc[qid, "top3"]
    print(f"\n{name}:")
    for p in top:
        print(f"  [{etiqueta[qrels_test[qid][p]]}] {titulos[p][:90]}")

### Resumen

- Las conclusiones finales se redactan con los números de test de este notebook ejecutado en Colab.
- Se guarda `data/index/retrieval_test_por_consulta.parquet` para el análisis de aciertos y fallos.
- El agente usará **híbrido → re-ranker** con `search(query, k)` sobre todo el catálogo.
- **Siguiente:** `04_agente.ipynb` (estado de la conversación, herramientas, validaciones y aclaraciones).